# RNA Structure Analysis

This notebook demonstrates how to work with RNA secondary structures using `seq_tools`.

## RNA Folding

RNA sequences can fold into secondary structures. The `fold` function uses ViennaRNA to predict the minimum free energy (MFE) structure.


In [7]:
from seq_tools import sequence_to_dataframe, fold
import pandas as pd

# Create a simple hairpin sequence
rna_seq = "GGGGUUUUCCCC"
df = sequence_to_dataframe(rna_seq, name="hairpin")

# Fold the sequence
df_folded = fold(df)

print("Sequence folding results:")
print(df_folded[["name", "sequence", "structure", "mfe", "ens_defect"]])

Sequence folding results:
      name      sequence     structure  mfe  ens_defect
0  hairpin  GGGGUUUUCCCC  ((((....)))) -5.9    0.378602


### Understanding the Output

- **structure**: Dot-bracket notation where `(` and `)` represent paired bases, and `.` represents unpaired bases
- **mfe**: Minimum free energy in kcal/mol (more negative = more stable)
- **ens_defect**: Ensemble defect (0-1 scale, lower is better, represents structural diversity)


In [8]:
# Example: Visualize the structure
sequences = [
    "GGGGUUUUCCCC",  # Simple hairpin
    "GCGAAAGC",  # Another hairpin
    "AUGCAUGCAUGC",  # Less structured
]

results = []
for seq in sequences:
    df = sequence_to_dataframe(seq, name=seq)
    df = fold(df)
    results.append(df)

df_results = pd.concat(results, ignore_index=True)

print("Multiple sequences folded:")
for _, row in df_results.iterrows():
    print(f"\nSequence: {row['sequence']}")
    print(f"Structure: {row['structure']}")
    print(f"MFE: {row['mfe']:.2f} kcal/mol")
    print(f"Ensemble defect: {row['ens_defect']:.3f}")

Multiple sequences folded:

Sequence: GGGGUUUUCCCC
Structure: ((((....))))
MFE: -5.90 kcal/mol
Ensemble defect: 0.379

Sequence: GCGAAAGC
Structure: ((....))
MFE: -0.10 kcal/mol
Ensemble defect: 0.993

Sequence: AUGCAUGCAUGC
Structure: ............
MFE: 0.00 kcal/mol
Ensemble defect: 1.849


## Working with Structure Columns

When you fold sequences, the structure information is stored as a column in the DataFrame. You can work with these structure strings directly for analysis.


In [ ]:
from seq_tools import sequence_to_dataframe, fold

# Create a sequence and fold it
seq = "GGGGUUUUCCCC"
df = sequence_to_dataframe(seq, name="example")
df = fold(df)

struct = df.iloc[0]["structure"]
print(f"Sequence: {seq}")
print(f"Structure: {struct}")
print(f"Length: {len(seq)}")

# Analyze structure manually
# Count base pairs (opening and closing parentheses)
open_parens = struct.count("(")
close_parens = struct.count(")")
print(f"\nNumber of opening parentheses: {open_parens}")
print(f"Number of closing parentheses: {close_parens}")
print(f"Number of base pairs: {open_parens}")

# Count unpaired bases (dots)
unpaired = struct.count(".")
print(f"Number of unpaired bases: {unpaired}")

# Find unpaired positions
unpaired_positions = [i for i, char in enumerate(struct) if char == "."]
print(f"Unpaired positions: {unpaired_positions}")

Sequence: GGGGUUUUCCCC
Structure: ((((....))))
Length: 12

Paired positions: [(3, 8), (2, 9), (1, 10), (0, 11)]
Number of base pairs: 4
Unpaired positions: [4, 5, 6, 7]
Number of unpaired bases: 4


## Searching for Structural Patterns

You can search for structural patterns within sequences using string matching on the structure column. This is useful for finding specific motifs or structural elements.


In [ ]:
from seq_tools import sequence_to_dataframe, fold
import re

# Target structure (a larger RNA)
target_seq = "GGGGUUUUCCCCAAAGGGGUUUUCCCC"
df_target = sequence_to_dataframe(target_seq, name="target")
df_target = fold(df_target)
target_struct = df_target.iloc[0]["structure"]

# Search for a pattern (hairpin structure)
pattern_struct = "((((....))))"

print(f"Target sequence: {target_seq}")
print(f"Target structure: {target_struct}")
print(f"\nPattern structure: {pattern_struct}")

# Find all matches using string search
# Escape special regex characters in the pattern
pattern_escaped = (
    pattern_struct.replace("(", r"\(").replace(")", r"\)").replace(".", r"\.")
)
matches = list(re.finditer(pattern_escaped, target_struct))

print(f"\nFound {len(matches)} match(es):")
for i, match in enumerate(matches, 1):
    start = match.start()
    end = match.end() - 1  # end is exclusive, so subtract 1
    print(f"\nMatch {i}:")
    print(f"  Positions: {start} to {end}")
    print(f"  Matched sequence: {target_seq[start:end+1]}")
    print(f"  Matched structure: {target_struct[start:end+1]}")

Target sequence: GGGGUUUUCCCCAAAGGGGUUUUCCCC
Target structure: ((((....))))...((((....))))

Pattern sequence: GGGGUUUUCCCC
Pattern structure: ((((....))))

Found 2 match(es):

Match 1:
  Positions: 15 to 27
  All strand positions: [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
  Matched sequence: GGGGUUUUCCCC

Match 2:
  Positions: 0 to 12
  All strand positions: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
  Matched sequence: GGGGUUUUCCCCA


## Structure-Aware Extinction Coefficient

When you have structure information, the extinction coefficient calculation can account for hypochromicity effects from base pairing.


In [11]:
from seq_tools import get_extinction_coeff

# Same sequence, with and without structure
rna_seq = "GGGGUUUUCCCC"
structure = "((((....))))"

# Without structure
ec_no_struct = get_extinction_coeff(rna_seq, "RNA", double_stranded=False)
print(f"Extinction coefficient (no structure): {ec_no_struct:,} M⁻¹cm⁻¹")

# With structure (accounts for hypochromicity)
ec_with_struct = get_extinction_coeff(
    rna_seq, "RNA", double_stranded=False, structure=structure
)
print(f"Extinction coefficient (with structure): {ec_with_struct:,} M⁻¹cm⁻¹")
print(f"\nDifference: {ec_no_struct - ec_with_struct:,} M⁻¹cm⁻¹")
print("(Structured RNA has lower extinction coefficient due to base pairing)")

Extinction coefficient (no structure): 109,500 M⁻¹cm⁻¹
Extinction coefficient (with structure): 105,193 M⁻¹cm⁻¹

Difference: 4,307 M⁻¹cm⁻¹
(Structured RNA has lower extinction coefficient due to base pairing)


## Analyzing Base Pairing

You can examine the pairing relationships in a structure by parsing the dot-bracket notation.


In [ ]:
from seq_tools import sequence_to_dataframe, fold

# Create a sequence and fold it
seq = "GGGGUUUUCCCC"
df = sequence_to_dataframe(seq, name="example")
df = fold(df)
struct = df.iloc[0]["structure"]

print(f"Sequence: {seq}")
print(f"Structure: {struct}")


# Parse base pairs from dot-bracket notation
def parse_base_pairs(structure):
    """Parse base pairs from dot-bracket notation."""
    stack = []
    pairs = []
    for i, char in enumerate(structure):
        if char == "(":
            stack.append(i)
        elif char == ")":
            if stack:
                j = stack.pop()
                pairs.append((j, i))
    return pairs


base_pairs = parse_base_pairs(struct)
print(f"\nBase pairs: {base_pairs}")

# Check which positions are paired
paired_positions = set()
for i, j in base_pairs:
    paired_positions.add(i)
    paired_positions.add(j)

print("\nPairing information:")
for i in range(len(seq)):
    if i in paired_positions:
        # Find the paired position
        for pair in base_pairs:
            if pair[0] == i:
                paired_idx = pair[1]
                print(
                    f"Position {i} ({seq[i]}) pairs with position {paired_idx} ({seq[paired_idx]}) -> {seq[i]}{seq[paired_idx]}"
                )
                break
            elif pair[1] == i:
                paired_idx = pair[0]
                print(
                    f"Position {i} ({seq[i]}) pairs with position {paired_idx} ({seq[paired_idx]}) -> {seq[paired_idx]}{seq[i]}"
                )
                break
    else:
        print(f"Position {i} ({seq[i]}) is unpaired")

Sequence: GGGGUUUUCCCC
Structure: ((((....))))

Connectivity list: [11, 10, 9, 8, -1, -1, -1, -1, 3, 2, 1, 0]

Pairing information:
Position 0 (G) pairs with position 11 (C) -> GC
Position 1 (G) pairs with position 10 (C) -> GC
Position 2 (G) pairs with position 9 (C) -> GC
Position 3 (G) pairs with position 8 (C) -> GC
Position 4 (U) is unpaired
Position 5 (U) is unpaired
Position 6 (U) is unpaired
Position 7 (U) is unpaired
Position 8 (C) pairs with position 3 (G) -> CG
Position 9 (C) pairs with position 2 (G) -> CG
Position 10 (C) pairs with position 1 (G) -> CG
Position 11 (C) pairs with position 0 (G) -> CG


## Summary

In this notebook, we've covered:

- ✅ RNA folding with ViennaRNA
- ✅ Understanding dot-bracket notation
- ✅ Working with structure columns in DataFrames
- ✅ Searching for structural patterns using string matching
- ✅ Structure-aware extinction coefficient calculations
- ✅ Analyzing pairing relationships by parsing dot-bracket notation

Next, we'll explore batch processing with DataFrames in **04_dataframe_operations.ipynb**.
